# Predicting nightly prices of Airbnb listings in London
Module: 6COSC017C-n Machine Learning and Data Analytics, CW1

Student ID: 00017795

The goal is to predict the nightly price of a listing from its size, location, amenities, host and review information.
Data: Inside Airbnb, London detailed listings (https://insideairbnb.com/get-the-data/).

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from xgboost import XGBRegressor

## 1. Load dataset

In [2]:
df = pd.read_csv('data/listings.csv', low_memory=False)
print("DataFrame's shape: ", df.shape)
df.head()

DataFrame's shape:  (91954, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,11551,https://www.airbnb.com/rooms/11551,20260917015706,2026-09-19,NaN,Artistic London Pied-à-Terre,Live like a local in this quintessential Londo...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,43039,...,4.84,4.54,4.49,NaN,NaN,1,1,0,0,0.98
1,13913,https://www.airbnb.com/rooms/13913,20260917015706,2026-09-19,NaN,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.88,4.79,4.79,NaN,NaN,2,1,1,0,0.29
2,15400,https://www.airbnb.com/rooms/15400,20260917015706,2026-09-19,NaN,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,NaN,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.84,4.94,4.74,NaN,NaN,1,1,0,0,0.48
3,24328,https://www.airbnb.com/rooms/24328,20260917015706,2026-09-19,NaN,Battersea live/work artist house with car park,"Artist house by SW Battersea Park, bright high...",NaN,https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,4.93,4.60,4.66,NaN,NaN,1,1,0,0,0.50
4,25081,https://www.airbnb.com/rooms/25081,20260917015706,2026-09-21,previous scrape,Lovely bright flat for 1-4p 20 m to Covent Garden,"<b>The space</b><br />Big, charming flat in We...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,72242,...,NaN,NaN,NaN,NaN,NaN,1,1,0,0,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 91954 entries, 0 to 91953
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            91954 non-null  int64  
 1   listing_url                                   91954 non-null  str    
 2   scrape_id                                     91954 non-null  int64  
 3   last_scraped                                  91954 non-null  str    
 4   source                                        32311 non-null  str    
 5   name                                          91954 non-null  str    
 6   description                                   91253 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   91949 non-null  str    
 9   host_id                                       91954 non-null  int64  
 1